# AI Medical Symptom Routing System — Full Pipeline
**NTI Machine Learning Training - Final Project (Merged Notebook)**

## Project Overview
This notebook contains the complete end-to-end pipeline for an AI-driven healthcare recommendation system, merging all project phases into a single, sequential workflow:

- **Phase 1 — Data Engineering & EDA** (Abdulrahman Jamal): load, clean, and explore the raw dataset; resolve data leakage; encode features; save the cleaned dataset.
- **Phase 2 — Machine Learning Modeling** (Marwan Ayman): hyperparameter tuning and training of a Multi-Output Random Forest Classifier; export the optimized model.
- **Phase 3 — Pretrained Model (TabPFN) + Stacking Ensemble** (Person 3): builds on the cleaned dataset from Phase 1 and the tuned parameters from Phase 2, adds TabPFN (a pretrained tabular foundation model), combines it with the RandomForest models via a Stacking Ensemble meta-model, chooses optimal decision thresholds, evaluates on a held-out test set, and includes an appendix investigating the (very high) evaluation scores for potential data leakage.

> **Note:** Run the notebook top to bottom. Phase 1 saves `leakage_free_chronic_dataset.csv`, which Phase 3 reloads (this keeps phase 3 fully reproducible even if run separately, but it means the cleaned dataset is written to disk mid-notebook).

In [ ]:
# Import all required libraries for the pipeline
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report
import joblib

sns.set_theme(style="whitegrid")
import warnings
warnings.filterwarnings('ignore')

---
# Phase 1: Data Engineering & EDA
**Responsible:** Abdulrahman Jamal

In this phase, we will load the raw dataset, clean it, engineer new features, and perform Exploratory Data Analysis (EDA) one step at a time.

## 1. Load Data

In [ ]:
df = pd.read_csv('Data Warehouse Multiclass.csv')
print(f"Initial Shape: {df.shape}")
display(df.head())

In [ ]:
display(df.info())

In [ ]:
display(df.describe())

In [ ]:
display(df.describe(include='O'))

**Observation:** 
The raw dataset is loaded successfully. However, a preliminary data audit revealed severe **Data Leakage** issues. Several columns are directly derived from the target variables, which would allow the ML model to "cheat" during training.

## 2. Data Cleaning & Sanitization
In this step, we will:
1. Check and remove duplicate rows to prevent the model from overfitting.
2. Fix logical anomalies in numerical fields (e.g., converting negative billing values to absolute positive amounts).
3. Resolve Data Leakage & Clean the Data.
Based on the data audit, we must drop columns that leak target information.

In [ ]:
leakage_columns = [
    'disease_flags',  # Derived directly from targets
    'label',          # Derived directly from targets
    'sublabel',       # Derived directly from targets
    'composite_key',  # Contains the diabetes indicator
    'source_dataset'  # Leaks patient status based on origin dataset
]

# Drop the leakage columns if they exist in the dataframe
df_cleaned = df.drop(columns=[col for col in leakage_columns if col in df.columns])

# Check for duplicates
duplicates_count = df_cleaned.duplicated().sum()
df_cleaned.drop_duplicates(inplace=True)

print(f"Removed Duplicates: {duplicates_count}")
print(f"Shape after resolving data leakage: {df_cleaned.shape}")

**Critical Inference (Data Leakage Resolution):** 
We successfully removed 5 critical columns (`disease_flags`, `label`, `sublabel`, `composite_key`, `source_dataset`). Keeping these would have resulted in artificially high model accuracy. 

*Note for Presentation:* The co-morbidity in this dataset is partly artificial (e.g., heart disease/hypertension values were imputed for patients sourced from the diabetes dataset). We must acknowledge this limitation when evaluating model performance.

In [ ]:
isnull_counts = df_cleaned.isnull().sum().sum()
print("Missing Values Count:\n", isnull_counts)

## 3. Exploratory Data Analysis (EDA)

*Note: Before plotting, we will convert the target variables into binary format so they can be analyzed correctly against our numeric features.*

### Graph 1: Target Distributions

In [ ]:
# We have 3 separate target columns
targets = ['heart_disease', 'hypertension', 'diabetes']

# Convert heart_disease and hypertension to binary (0/1) format like diabetes
df_cleaned['heart_disease'] = (df_cleaned['heart_disease'] > 0).astype(int)
df_cleaned['hypertension'] = (df_cleaned['hypertension'] > 0).astype(int)
df_cleaned['diabetes'] = (df_cleaned['diabetes'] == 'Yes').astype(int)

print("Targets converted to binary format:")
display(df_cleaned[targets].head(10))

plt.figure(figsize=(12, 5))
for i, target in enumerate(targets, 1):
    plt.subplot(1, 3, i)
    sns.countplot(data=df_cleaned, x=target, palette='viridis')
    plt.title(f'{target.replace("_", " ").title()} Distribution')
    plt.ylabel('Count')

plt.tight_layout()
plt.show()

**Inference from Graph 1:** 
By visualizing our three distinct targets (`heart_disease`, `hypertension`, `diabetes`), we can assess the class balance for each specific disease. If any of these classes are heavily imbalanced (e.g., 90% zeros and 10% ones), the ML Engineer will need to apply balancing techniques (like SMOTE or class weighting) during Phase 2.

### Graph 2: Correlation Heatmap

In [ ]:
plt.figure(figsize=(14, 10))

# Calculate correlation matrix for numeric columns only
corr_matrix = df_cleaned.select_dtypes(include=[np.number]).corr()

# Plot the heatmap
sns.heatmap(corr_matrix, annot=False, cmap='coolwarm', linewidths=0.5)
plt.title('Correlation Heatmap of Features and Targets', fontsize=16)
plt.show()

**Inference from Correlation Heatmap:** 
This heatmap visualizes the linear relationships between all numerical features and our three target diseases. 
*   **Feature-Target Correlation:** Dark red or dark blue spots between a feature and a target indicate a strong positive or negative correlation, making that feature a strong predictor.
*   **Multicollinearity:** If two features are highly correlated with each other (e.g., above 0.8), we might consider dropping one of them later to simplify the model and avoid multicollinearity.

### Graph 3: Feature Distribution by Targets

We will dynamically pick the first two continuous numeric features to see how they relate to the diseases.

In [ ]:
numeric_features = df_cleaned.select_dtypes(include=[np.number]).columns.drop(targets, errors='ignore')

if len(numeric_features) >= 2:
    feature_to_plot = numeric_features[0] # it can be changed to any other numeric feature you want to visualize against the targets
    
    plt.figure(figsize=(15, 5))
    for i, target in enumerate(targets, 1):
        plt.subplot(1, 3, i)
        sns.boxplot(data=df_cleaned, x=target, y=feature_to_plot, palette='Set2')
        plt.title(f'{feature_to_plot} vs {target.replace("_", " ").title()}')
        plt.xlabel(f'Has {target.title()}? (0=No, 1=Yes)')
    
    plt.tight_layout()
    plt.show()
else:
    print("Not enough numeric features for boxplots.")

**Inference from Feature Distributions:** 
These boxplots illustrate how a specific continuous feature varies between patients who have the disease (1) and those who do not (0). 
*   If the boxes (median and interquartile ranges) are at significantly different levels on the Y-axis, it implies that this feature strongly influences whether a patient gets the disease or not. 
*   It also helps us spot potential **outliers** in the data that might need further cleaning.

### Graph 4: Skewness Check & Feature Transformation

Before feature selection, it is important to check the distribution of continuous numerical features. Highly skewed data (skewness > 1 or < -1) can degrade the performance of certain machine learning algorithms and typically requires mathematical transformations (e.g., Log1p or Box-Cox) to stabilize the variance. 

In a complex, real-time multi-model architecture, keeping the preprocessing pipeline lean is critical for maintaining low inference latency. We will calculate the skewness of our continuous features and visualize a few key distributions to determine if transformations are strictly necessary.

In [ ]:
# Identify continuous columns
continuous_cols = [
    'age', 'bmi', 'HbA1c_level', 'glucose', 'cholesterol', 'sleep_hours', 
    'triglycerides', 'blood_pressure', 'crp_level', 'homocysteine_level', 
    'systolic_bp', 'diastolic_bp', 'alcohol_intake', 'salt_intake', 
    'heart_rate', 'hdl', 'ldl'
]

# Calculate skewness for continuous features
# Note: Skewness between -0.5 and 0.5 indicates a fairly symmetrical distribution.
skewness = df_cleaned[continuous_cols].skew().sort_values(ascending=False)

print("--- Feature Skewness Scores ---")
print(skewness)
print("\nConclusion: All continuous features have a skewness score between -0.5 and 0.5.")
print("The data is highly symmetrical. No mathematical transformations (like Log1p) are required.")

# Visualize a few key distributions to confirm visually
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

sns.histplot(df_cleaned['glucose'], kde=True, ax=axes[0], color='skyblue')
axes[0].set_title('Glucose Distribution')

sns.histplot(df_cleaned['bmi'], kde=True, ax=axes[1], color='salmon')
axes[1].set_title('BMI Distribution')

sns.histplot(df_cleaned['triglycerides'], kde=True, ax=axes[2], color='lightgreen')
axes[2].set_title('Triglycerides Distribution')

plt.tight_layout()
plt.show()

## 4. Feature Selection & Multicollinearity Handling

In this step, we will drop redundant features that represent the same underlying data to prevent multicollinearity (which can confuse the model and artificially inflate feature importance).

In [ ]:
# List of redundant columns to drop based on correlation insights
redundant_columns = [
    'age_normalized',       # Redundant with 'age'
    'age_level',            # Redundant with 'age'
    'bmi_level',            # Redundant with 'bmi'
    'blood_pressure',       # Redundant with 'systolic_bp' and 'diastolic_bp'
    'high_blood_pressure'   # Redundant with the continuous bp features
]

# Drop the columns
df_cleaned = df_cleaned.drop(columns=[col for col in redundant_columns if col in df_cleaned.columns])

print(f"Shape after removing redundant features: {df_cleaned.shape}")

## 5. Categorical Variable Encoding
Machine learning models require numerical inputs. We will encode our string variables using three methods:
1. **Binary Encoding:** For Yes/No or Male/Female features.
2. **Ordinal Encoding:** For categorical variables that have a natural ranking (e.g., Low, Moderate, High).
3. **One-Hot Encoding:** For nominal variables without an inherent ranking (e.g., employment status).

In [ ]:
# 1. Binary Encoding
binary_mapping = {'Yes': 1, 'No': 0}
binary_cols = ['family_history', 'low_hdl_cholesterol', 'high_ldl_cholesterol']

for col in binary_cols:
    if col in df_cleaned.columns:
        df_cleaned[col] = df_cleaned[col].map(binary_mapping)

if 'gender' in df_cleaned.columns:
    df_cleaned['gender'] = df_cleaned['gender'].map({'Male': 1, 'Female': 0})

# 2. Ordinal Encoding
# Mapping for Low/Moderate/High
ordinal_mapping = {'Low': 0, 'Moderate': 1, 'Medium': 1, 'High': 2} 
ordinal_cols = ['physical_activity', 'stress_level', 'sugar_consumption']

for col in ordinal_cols:
    if col in df_cleaned.columns:
        df_cleaned[col] = df_cleaned[col].map(ordinal_mapping)

# Mapping for Education Level
if 'education_level' in df_cleaned.columns:
    edu_mapping = {'Primary': 0, 'Secondary': 1, 'Tertiary': 2}
    df_cleaned['education_level'] = df_cleaned['education_level'].map(edu_mapping)

# 3. One-Hot Encoding for remaining nominal categorical variables
# Using drop_first=True to avoid the dummy variable trap
nominal_cols = ['smoking', 'employment_status']
df_cleaned = pd.get_dummies(df_cleaned, columns=[col for col in nominal_cols if col in df_cleaned.columns], drop_first=True, dtype=int)

print(f"Shape after all encoding: {df_cleaned.shape}")

# Verify that all columns are now numeric
print("\\nData Types after Encoding:")
display(df_cleaned.dtypes.value_counts())
display(df_cleaned.head())

In [ ]:
# Correlation map between all features (convert non-numeric to categorical codes first)
corr_df = df_cleaned.copy()

for col in corr_df.columns:
    if not np.issubdtype(corr_df[col].dtype, np.number):
        corr_df[col] = pd.Categorical(corr_df[col]).codes

corr_all = corr_df.corr(method='spearman')

plt.figure(figsize=(20, 16))
mask = np.triu(np.ones_like(corr_all, dtype=bool))
sns.heatmap(corr_all,  cmap='coolwarm', center=0, linewidths=0.5, cbar_kws={'shrink':0.6})
plt.title('Correlation Map — All Features', fontsize=16)
plt.tight_layout()
plt.show()

## 6. Save the Preprocessed Dataset

In [ ]:
df_cleaned.to_csv('leakage_free_chronic_dataset.csv', index=False)
display(df_cleaned.head())
print(f"Cleaned Shape: {df_cleaned.shape}")
print("✅ Data sanitized, leakage resolved, and saved successfully as 'leakage_free_chronic_dataset.csv'")

---
# Phase 2: Machine Learning Modeling
**Responsible:** Marwan Ayman

In this phase, we will separate our features from our targets, split the data into training and testing sets, and train a Multi-Output Random Forest Classifier to predict all three medical conditions simultaneously.

## 1. Hyperparameter Tuning
To squeeze the highest possible accuracy and recall out of our model, we will use `GridSearchCV` to find the optimal hyperparameters. 

Since our `RandomForestClassifier` is wrapped inside a `MultiOutputClassifier`, we must use the `estimator__` prefix for our parameter grid keys so the grid search knows to apply them to the underlying Random Forest.

In [ ]:
from sklearn.multioutput import MultiOutputClassifier

# 1. Define Features (X) and Targets (y)
targets = ['heart_disease', 'hypertension', 'diabetes']
X = df_cleaned.drop(columns=targets)
y = df_cleaned[targets]

# 2. Train-Test Split (80% training, 20% testing)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Define the base model and multi-output wrapper
base_rf = RandomForestClassifier(class_weight='balanced', random_state=42)
multi_target_rf = MultiOutputClassifier(base_rf, n_jobs=-1)

# 4. Define the Parameter Grid
param_grid = {
    'estimator__n_estimators': [100, 150],
    'estimator__max_depth': [10, 15, 20],
    'estimator__min_samples_split': [2, 5]
}

# 5. Initialize GridSearchCV
# We use scoring='f1_macro' to prioritize balancing precision and recall due to class imbalance
grid_search = GridSearchCV(
    estimator=multi_target_rf,
    param_grid=param_grid,
    cv=3, 
    n_jobs=-1,
    verbose=2,
    scoring='f1_macro' 
)

print("Starting Grid Search... (This may take some time depending on your hardware)")
grid_search.fit(X_train, y_train)

print(f"\\nBest Parameters Found: {grid_search.best_params_}")

## 2. Optimized Model Evaluation
Now that we have found the optimal hyperparameters, we will extract the best estimator from our grid search and evaluate its performance on the unseen test data.

In [ ]:
# Extract the best model from the grid search
best_multi_target_rf = grid_search.best_estimator_

# Generate predictions
y_pred_best = best_multi_target_rf.predict(X_test)

# Evaluate each target separately
for i, target in enumerate(targets):
    print(f"\\n{'='*40}")
    print(f"Optimized Classification Report: {target.replace('_', ' ').title()}")
    print(f"{'='*40}")
    print(classification_report(y_test.iloc[:, i], y_pred_best[:, i]))

## 3. Export the Optimized Model for Deployment
**Responsible:** Hassan Raafat

We will save the optimized multi-output model to disk so it can be loaded into our local web server using Streamlit.

In [ ]:
# Save the trained model to a file
model_filename = 'optimized_multi_target_medical_model.pkl'
joblib.dump(best_multi_target_rf, model_filename)

print(f"✅ Optimized model saved successfully as '{model_filename}'")

---
---
# Phase 3 begins below

*(Continuing from `leakage_free_chronic_dataset.csv`, produced at the end of Phase 1, and the tuned hyperparameters used in Phase 2.)*

---
# Phase 3: Pretrained Model (TabPFN) + Stacking Ensemble
**Responsible:** Person 3

This notebook takes the cleaned dataset from Person 1 and the tuned model from Person 2,
combines it with TabPFN (a pretrained tabular foundation model) using a Stacking Ensemble,
and produces the final risk predictions for Diabetes, Heart Disease, and Hypertension.
---

## 0. Imports

In [ ]:
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import (
    classification_report, roc_auc_score, average_precision_score,
    precision_recall_curve, confusion_matrix, f1_score, accuracy_score
)

import warnings
warnings.filterwarnings('ignore')

## 1. Load Data & Artifacts from Person 1 & Person 2

In [ ]:
# Cleaned dataset from Person 1
df = pd.read_csv('leakage_free_chronic_dataset.csv')

targets = ['heart_disease', 'hypertension', 'diabetes']
X = df.drop(columns=targets)
y = df[targets]

print(f"Features shape: {X.shape}")
print(f"Targets shape: {y.shape}")

## 2. Remove Exact Duplicate Rows (Fix: Train/Test Contamination)

Before splitting, we check for exact duplicate rows in the feature set. If duplicates exist,
some of them can end up in both the train and test sets after a random split, which means the
model would effectively "see" test rows during training. We remove duplicates first to guarantee
a clean, non-overlapping split.


In [ ]:
duplicates_count = X.duplicated().sum()
print(f"Exact duplicate rows in X: {duplicates_count} ({duplicates_count / len(X) * 100:.2f}%)")

df = df.drop_duplicates().reset_index(drop=True)
X = df.drop(columns=targets)
y = df[targets]

print(f"Shape after removing duplicates: {X.shape}")

## 3. Recreate the Train/Test Split

We use the same `random_state` and `test_size` as Person 2, so the test set stays consistent
and is never touched during training or hyperparameter tuning.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

# Sanity check: confirm there is no overlap between train and test after deduplication
train_set = set(map(tuple, X_train.values))
test_set = set(map(tuple, X_test.values))
overlap = train_set.intersection(test_set)
print(f"Overlapping rows between Train and Test: {len(overlap)}")

## 4. Generate Out-of-Fold (OOF) Predictions — Our Models

Instead of the single Multi-Output model Person 2 built, we train one RandomForest per disease
(as originally planned) and extract probabilities via cross-validation on the training set only.

Why: if we used a model that was already fit on the full `X_train` to generate predictions on
`X_train`, the meta-model would be trained on leaked information. The fix is Out-of-Fold (OOF)
predictions — each row gets a prediction from a model that never saw that row during training.


In [ ]:
best_params = {
    'n_estimators': 150,
    'max_depth': 15,
    'min_samples_split': 2,
    'class_weight': 'balanced',
    'random_state': 42
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

our_oof_proba = pd.DataFrame(index=X_train.index)
our_models = {}

for target in targets:
    rf = RandomForestClassifier(**best_params)

    oof_proba = cross_val_predict(
        rf, X_train, y_train[target],
        cv=cv, method='predict_proba', n_jobs=-1
    )[:, 1]  # probability of class 1

    our_oof_proba[f'our_model_{target}'] = oof_proba

    # Refit on the full training set so we can use it later on the test set
    rf_final = RandomForestClassifier(**best_params)
    rf_final.fit(X_train, y_train[target])
    our_models[target] = rf_final

print("OOF predictions generated for our models:")
display(our_oof_proba.head())

## 5. Install & Authenticate TabPFN

In [ ]:
!pip install tabpfn --quiet

from tabpfn import TabPFNClassifier

print(f"Train size: {X_train.shape[0]} rows, {X_train.shape[1]} features")

TabPFN requires a one-time license acceptance and an API token:

1. Open https://ux.priorlabs.ai and log in / register
2. Accept the license under the **Licenses** tab
3. Copy your API key from https://ux.priorlabs.ai/account
4. Paste it below when prompted (input is hidden, this is safer than hardcoding the token in the notebook)


In [ ]:
import os
from getpass import getpass

os.environ["TABPFN_TOKEN"] = getpass("Enter your TabPFN API Token: ")



## 6. Out-of-Fold (OOF) Predictions — TabPFN

In [ ]:
tabpfn_oof_proba = pd.DataFrame(index=X_train.index)

for target in targets:
    oof_col = np.zeros(len(X_train))

    for train_idx, val_idx in cv.split(X_train, y_train[target]):
        X_fold_train = X_train.iloc[train_idx]
        y_fold_train = y_train[target].iloc[train_idx]
        X_fold_val = X_train.iloc[val_idx]

        clf = TabPFNClassifier()
        clf.fit(X_fold_train.values, y_fold_train.values)
        proba = clf.predict_proba(X_fold_val.values)[:, 1]

        oof_col[val_idx] = proba

    tabpfn_oof_proba[f'tabpfn_{target}'] = oof_col
    print(f"TabPFN OOF done for {target}")

display(tabpfn_oof_proba.head())

## 7. Train Final TabPFN Models on the Full Training Set

In [ ]:
tabpfn_models = {}

for target in targets:
    clf = TabPFNClassifier()
    clf.fit(X_train.values, y_train[target].values)
    tabpfn_models[target] = clf

print("Final TabPFN models trained on full training set")

## 8. Build Meta-Features DataFrame

In [ ]:
meta_features_train = pd.concat([our_oof_proba, tabpfn_oof_proba], axis=1)

# Organize columns per disease
meta_features_train = meta_features_train[[
    'our_model_diabetes', 'tabpfn_diabetes',
    'our_model_heart_disease', 'tabpfn_heart_disease',
    'our_model_hypertension', 'tabpfn_hypertension'
]]

display(meta_features_train.head())
print(f"Meta-features shape: {meta_features_train.shape}")

## 9. Train the Meta Model (Stacking)

We use a simple Logistic Regression as the meta-model — it learns the best weight to give
each base model (ours vs. TabPFN) for each disease.


In [ ]:
meta_models = {}

for target in targets:
    cols = [f'our_model_{target}', f'tabpfn_{target}']

    meta_clf = LogisticRegression()
    meta_clf.fit(meta_features_train[cols], y_train[target])

    meta_models[target] = meta_clf

    coef = meta_clf.coef_[0]
    print(f"{target}: our_model weight={coef[0]:.3f}, tabpfn weight={coef[1]:.3f}")

## 10. Choose Optimal Thresholds (Using OOF Predictions)

Instead of an arbitrary 0.5 cutoff, we pick the threshold that maximizes F1 on the OOF
predictions, since in a healthcare context missing a true positive (false negative) is generally
more costly than a false alarm.


In [ ]:
thresholds = {}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, target in enumerate(targets):
    cols = [f'our_model_{target}', f'tabpfn_{target}']
    meta_proba = meta_models[target].predict_proba(meta_features_train[cols])[:, 1]

    precision, recall, thresh = precision_recall_curve(y_train[target], meta_proba)

    f1_scores = 2 * (precision * recall) / (precision + recall + 1e-8)
    best_idx = np.argmax(f1_scores)
    best_threshold = thresh[best_idx] if best_idx < len(thresh) else 0.5

    thresholds[target] = best_threshold

    axes[i].plot(recall, precision)
    axes[i].scatter(recall[best_idx], precision[best_idx], color='red', zorder=5)
    axes[i].set_title(f'{target}\nThreshold={best_threshold:.2f}')
    axes[i].set_xlabel('Recall')
    axes[i].set_ylabel('Precision')

plt.tight_layout()
plt.show()

print("Selected thresholds:", thresholds)

## 11. Final Evaluation on Test Set (Never Seen Before)

In [ ]:
# Predictions from our model on the test set
our_test_proba = pd.DataFrame(index=X_test.index)
for target in targets:
    our_test_proba[f'our_model_{target}'] = our_models[target].predict_proba(X_test)[:, 1]

# Predictions from TabPFN on the test set
tabpfn_test_proba = pd.DataFrame(index=X_test.index)
for target in targets:
    tabpfn_test_proba[f'tabpfn_{target}'] = tabpfn_models[target].predict_proba(X_test.values)[:, 1]

meta_features_test = pd.concat([our_test_proba, tabpfn_test_proba], axis=1)

final_results = {}

for target in targets:
    cols = [f'our_model_{target}', f'tabpfn_{target}']
    final_proba = meta_models[target].predict_proba(meta_features_test[cols])[:, 1]
    final_pred = (final_proba >= thresholds[target]).astype(int)

    print(f"\n{'='*50}")
    print(f"Final Evaluation: {target.replace('_', ' ').title()}")
    print(f"{'='*50}")
    print(classification_report(y_test[target], final_pred))
    print(f"ROC-AUC: {roc_auc_score(y_test[target], final_proba):.3f}")
    print(f"PR-AUC:  {average_precision_score(y_test[target], final_proba):.3f}")

    cm = confusion_matrix(y_test[target], final_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f'Confusion Matrix - {target}')
    plt.show()

    final_results[target] = {
        'proba': final_proba,
        'pred': final_pred,
        'threshold': thresholds[target]
    }

## 12. Save Final Pipeline & Artifacts (for Person 4)

In [ ]:
joblib.dump(our_models, 'our_models.pkl')
joblib.dump(tabpfn_models, 'tabpfn_models.pkl')
joblib.dump(meta_models, 'meta_models.pkl')
joblib.dump(thresholds, 'thresholds.pkl')

print("All artifacts saved successfully:")
print("- our_models.pkl")
print("- tabpfn_models.pkl")
print("- meta_models.pkl")
print("- thresholds.pkl")

## 13. Download Artifacts as a Zip File

In [ ]:
from google.colab import files
import shutil, os

os.makedirs('person3_artifacts', exist_ok=True)

for f in ['our_models.pkl', 'tabpfn_models.pkl', 'meta_models.pkl', 'thresholds.pkl']:
    shutil.copy(f, f'person3_artifacts/{f}')

shutil.make_archive('person3_artifacts', 'zip', 'person3_artifacts')

files.download('person3_artifacts.zip')

---
## Appendix: Data Leakage Investigation

The final evaluation above shows near-perfect scores (ROC-AUC close to 1.0), which is unusual
for a real-world medical dataset. Before trusting these results, we investigate whether this is
caused by genuine leakage or by properties of the (partly synthetic) dataset itself, as flagged
earlier by Person 1.
---

### A1. Feature Ablation Test — With vs. Without Critical Features

We test whether `HbA1c_level` and `glucose` (the two most direct clinical indicators of diabetes)
are single-handedly responsible for the perfect score, by training the model with and without them.


In [ ]:
critical_features = ['HbA1c_level', 'glucose']

# With all features
X_train_with = X_train.copy()
X_test_with = X_test.copy()

rf_with = RandomForestClassifier(**best_params)
rf_with.fit(X_train_with, y_train['diabetes'])
proba_with = rf_with.predict_proba(X_test_with)[:, 1]
auc_with = roc_auc_score(y_test['diabetes'], proba_with)

# Without the critical features
X_train_without = X_train.drop(columns=critical_features)
X_test_without = X_test.drop(columns=critical_features)

rf_without = RandomForestClassifier(**best_params)
rf_without.fit(X_train_without, y_train['diabetes'])
proba_without = rf_without.predict_proba(X_test_without)[:, 1]
auc_without = roc_auc_score(y_test['diabetes'], proba_without)

print("="*50)
print("Diabetes Model Performance Comparison")
print("="*50)
print(f"With HbA1c_level and glucose    : ROC-AUC = {auc_with:.4f}")
print(f"Without HbA1c_level and glucose : ROC-AUC = {auc_without:.4f}")
print(f"Difference: {auc_with - auc_without:.4f}")

print("\n--- Classification report: WITH critical features ---")
print(classification_report(y_test['diabetes'], (proba_with >= 0.5).astype(int)))

print("\n--- Classification report: WITHOUT critical features ---")
print(classification_report(y_test['diabetes'], (proba_without >= 0.5).astype(int)))

importances = pd.Series(rf_without.feature_importances_, index=X_train_without.columns)
importances = importances.sort_values(ascending=False)

print("\n--- Top 10 real risk factors (excluding HbA1c/glucose) ---")
print(importances.head(10))

**Finding:** removing the two most obvious diabetes indicators makes no difference
(ROC-AUC stays at 1.0000). So the leakage, if any, is not caused by those two columns alone —
the separability is coming from somewhere else in the feature set.


### A2. Single-Feature Separability Check

We check whether any single remaining feature, on its own, can almost perfectly separate the
two classes.


In [ ]:
single_feature_auc = {}

for col in X_train_without.columns:
    try:
        auc = roc_auc_score(y_train['diabetes'], X_train_without[col])
        single_feature_auc[col] = max(auc, 1 - auc)
    except Exception:
        single_feature_auc[col] = np.nan

result = pd.Series(single_feature_auc).sort_values(ascending=False)
print("=== AUC of each individual feature vs. Diabetes ===")
print(result.head(15))

**Finding:** no single feature comes close to separating the classes on its own (best AUC
~0.64). So the perfect score is not caused by one dominant leaking column — it must come from
a combination of features.


### A3. Exact Duplicate Rows Check

We already removed exact duplicates before splitting (Section 2). Here we re-confirm there is
no remaining overlap between train and test.


In [ ]:
duplicates_count = X.duplicated().sum()
print(f"Exact duplicate rows remaining in X: {duplicates_count} ({duplicates_count / len(X) * 100:.2f}%)")

unique_rows = X.drop_duplicates()
print(f"Unique rows: {len(unique_rows)} out of {len(X)}")

train_set = set(map(tuple, X_train.values))
test_set = set(map(tuple, X_test.values))
overlap = train_set.intersection(test_set)
print(f"Rows shared between Train and Test: {len(overlap)} ({len(overlap) / len(X_test) * 100:.2f}% of test)")

### A4. Categorical Combination Determinism Check

We group patients by their categorical feature combination and check how often the diabetes
outcome is fully deterministic (always 0 or always 1) within a given combination.


In [ ]:
cat_features = ['gender', 'physical_activity', 'family_history', 'stress_level',
                 'sugar_consumption', 'education_level',
                 'smoking_Former', 'smoking_Never',
                 'employment_status_Retired', 'employment_status_Unemployed']

grouped = df.groupby(cat_features)['diabetes'].agg(['mean', 'count'])

print(f"Unique categorical combinations: {len(grouped)}")

deterministic = grouped['mean'].isin([0.0, 1.0]).sum()
print(f"Fully deterministic combinations (outcome always 0 or always 1): {deterministic} out of {len(grouped)}")
print(f"Percentage deterministic: {deterministic / len(grouped) * 100:.2f}%")

print(grouped['mean'].describe())

**Finding:** only ~5.6% of categorical combinations are fully deterministic — not nearly
enough on its own to explain a perfect model score.


### A5. Decision Tree Depth vs. Performance

We check how model complexity (tree depth) affects performance. If a very shallow, human-readable
tree already achieves near-perfect separation, that would point to simple leakage. If performance
only approaches 1.0 with a deep, complex model, that suggests the pattern is real but subtle
rather than a single obvious leak.


In [ ]:
shallow_tree = DecisionTreeClassifier(max_depth=3, random_state=42)
shallow_tree.fit(X_train_without, y_train['diabetes'])

proba = shallow_tree.predict_proba(X_test_without)[:, 1]
pred = shallow_tree.predict(X_test_without)

print(f"Accuracy: {accuracy_score(y_test['diabetes'], pred):.4f}")
print(f"ROC-AUC: {roc_auc_score(y_test['diabetes'], proba):.4f}")

print("\n=== Rules learned by the depth-3 tree ===")
print(export_text(shallow_tree, feature_names=list(X_train_without.columns)))

In [ ]:
for depth in [1, 2, 3, 4, 5, 6, 8, 10, 12, 15]:
    tree = DecisionTreeClassifier(max_depth=depth, random_state=42)
    tree.fit(X_train_without, y_train['diabetes'])
    auc = roc_auc_score(y_test['diabetes'], tree.predict_proba(X_test_without)[:, 1])
    print(f"Depth {depth}: ROC-AUC = {auc:.4f}")

### Conclusion of the Leakage Investigation

- A depth-3 tree only reaches ROC-AUC = 0.81, and even a depth-12 tree only reaches ~0.995 — not
  perfect. Performance keeps climbing as complexity increases, which means the classes are
  separable through a **subtle interaction across many features**, not through one obvious
  leaking column or a shallow rule.
- Exact duplicate rows (0.25% of the data) were removed before splitting, and train/test overlap
  was confirmed to be 0 afterward — that source of leakage is fixed.
- No single feature, and no simple categorical grouping, comes close to explaining the
  near-perfect separability on its own.

**Most likely explanation:** this lines up with Person 1's earlier note that the dataset was
assembled by combining multiple source datasets (diabetes / heart disease / hypertension
cohorts), with some fields imputed rather than measured for patients outside their original
source dataset. Even without an explicit "source" column, complex models (Random Forest, TabPFN)
are able to pick up on the residual statistical fingerprint of the original source dataset through
the *combination* of features — which is a form of dataset-level leakage that cannot be fully
removed by dropping any single column.

**Recommendation for the presentation:** report this transparently as a documented dataset
limitation rather than as a modeling success. It's worth stating explicitly that the near-perfect
scores likely overstate real-world performance, and that evaluating this pipeline on an
independent, non-synthetic clinical dataset would be necessary before drawing conclusions about
real-world accuracy.
